# Prepare the population shear-tilt inputs
Run once on Katana, then run **01_shear_tilt_climatology.ipynb**. No individual case selection and no new ROMS velocity/temperature processing. Reuses the v4 background-flow cache and authoritative coherent `TiltDis`/`TiltDir`.

The current **ESP-Gaussian, nonlinear, FRAC=1** environmental PV-gradient method supplies the regime diagnostic. This does not use `PV_grad_full` or presume that gradient dominance proves dynamical control. The output retains every eddy-day, including missing background records, so later temporal analysis can detect gaps and invalid periods.

Prerequisite: `beta_effect_background_flow/01_build_background_cache.ipynb` has produced `eddy_day_background.parquet`. The first run here computes surface PV gradients across all eddy-days; it can take time. Subsequent runs reuse the completed input cache only when settings, source files and helper hashes match.

In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Launch from seacofs_eddy_tilt_analysis or this subfolder.')
THIS_ROOT = ANALYSIS_ROOT / 'shear_tilt_climatology'
for p in (ANALYSIS_ROOT, THIS_ROOT):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
import climatology_tools as ct
CACHE_ROOT = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/shear_tilt_climatology')
INPUT_PATH = CACHE_ROOT / 'climatology_inputs.parquet'
METADATA_PATH = CACHE_ROOT / 'input_metadata.json'
pd.set_option('display.max_columns', 40)
plt.style.use('seaborn-v0_8-whitegrid')

In [2]:
import seacofs_tilt_tools as tilt
paths = tilt.Paths()
BACKGROUND_PATH = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/background_flow_cache_all_eddies_v4/eddy_day_background.parquet')
FORCE_REBUILD = False
# Optional source table produced by this same method; leave None for calculation.
PV_OPTIONS = dict(core_mean=True, frac=1.0, surface_method='esp_gaussian', averaging='nonlinear')
def fingerprint(path):
    path = Path(path)
    if not path.exists(): raise FileNotFoundError(path)
    files = sorted(path.rglob('*.parquet')) if path.is_dir() else [path]
    return [dict(path=str(p.resolve()), size=p.stat().st_size, mtime_ns=p.stat().st_mtime_ns) for p in files]
source_paths = [paths.eddies, paths.tilt, paths.grid, paths.z_r, BACKGROUND_PATH]
signature = dict(schema=1, pv=PV_OPTIONS, sources=[fingerprint(p) for p in source_paths],
    helper_sha256=hashlib.sha256((ANALYSIS_ROOT/'seacofs_tilt_tools.py').read_bytes()).hexdigest())
reuse = INPUT_PATH.exists() and METADATA_PATH.exists() and not FORCE_REBUILD
if reuse:
    reuse = json.loads(METADATA_PATH.read_text()).get('signature') == signature
print('Reusing verified input cache' if reuse else 'Building inputs from source tables')

Building inputs from source tables


In [3]:
if reuse:
    data = pd.read_parquet(INPUT_PATH)
else:
    grid = tilt.load_grid(paths.grid, paths.z_r)
    data, _ = tilt.load_tilt_tables(paths)
    ct.validate_days(data)
    background = pd.read_parquet(BACKGROUND_PATH)
    flow_cols = [f'{f}_{z}_{axis}_ms' for f in ('clim','full')
                 for z in ('surface','200','500') for axis in ('east','north')]
    missing = set(flow_cols + ['Eddy','Day']) - set(background)
    if missing: raise ValueError(f'Background cache lacks {sorted(missing)}; rebuild v4 cache.')
    if background.duplicated(['Eddy','Day']).any():
        raise ValueError('Background cache keys are not unique.')
    # Existing table uses globally unique Eddy identifiers; validate that contract.
    if data.duplicated(['Eddy','Day']).any():
        raise ValueError('Source uses polarity-local IDs; resolve background merge keys before proceeding.')
    data = tilt.add_pv_gradient_terms(data, grid, **PV_OPTIONS)
    # Reject invalid model indices instead of clipping to unrelated grid cells.
    ii = pd.to_numeric(data.ic, errors='coerce'); jj = pd.to_numeric(data.jc, errors='coerce')
    valid_idx = (ii.notna() & jj.notna() & ii.eq(ii.round()) & jj.eq(jj.round()) &
                 ii.between(0,grid.h.shape[0]-1) & jj.between(0,grid.h.shape[1]-1))
    data['lat'] = np.nan; data['water_depth_m'] = np.nan
    i = ii[valid_idx].astype(int).to_numpy(); j = jj[valid_idx].astype(int).to_numpy()
    data.loc[valid_idx, 'lat'] = grid.lat_rho[i,j]
    data.loc[valid_idx, 'water_depth_m'] = grid.h[i,j]
    data = data.drop(columns=flow_cols, errors='ignore').merge(
        background[['Eddy','Day']+flow_cols], on=['Eddy','Day'], how='left', validate='one_to_one', indicator='background_match')
    required = ['Cyc','Eddy','Day','TiltDis','TiltDir','h','water_depth_m','lat',
                'PV_grad_plan_mag','PV_grad_topo_mag','PV_grad_mag','PV_grad_theta']
    optional = [c for c in ['Rc','w','Ro','Region','PV_grad_coherence','PV_footprint_n'] if c in data]
    data = data[required + optional + flow_cols + ['background_match']].sort_values(ct.KEYS+['Day'])
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    temporary = INPUT_PATH.with_suffix('.tmp.parquet')
    data.to_parquet(temporary, index=False); temporary.replace(INPUT_PATH)
    metadata = dict(signature=signature, rows=len(data), eddies=len(data[ct.KEYS].drop_duplicates()),
                    created_utc=pd.Timestamp.now(tz='UTC').isoformat(),
                    tilt='authoritative coherent bottom-to-surface; not matched layer centroids',
                    background='91-day moving monthly climatology and full-archive means; layer means at surface centre')
    temp_meta = METADATA_PATH.with_suffix('.tmp.json')
    temp_meta.write_text(json.dumps(metadata, indent=2)); temp_meta.replace(METADATA_PATH)
ct.validate_days(data)
display(data.groupby('Cyc').agg(days=('Day','size'), eddies=('Eddy','nunique'),
    finite_tilt=('TiltDir','count'), median_depth=('water_depth_m','median')))
display(data.groupby(['Cyc','background_match'], observed=True).size().rename('days').to_frame())
print(INPUT_PATH)

,days,eddies,finite_tilt,median_depth
Cyc,,,,
AE,64952,1446,53607,4629.417227
CE,62474,1536,52014,4572.220750


,,days
Cyc,background_match,
AE,both,64952
CE,both,62474


/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/shear_tilt_climatology/climatology_inputs.parquet


## What this cache does not establish
The background is a seasonal climatology, not a contemporaneous eddy-free velocity field. Core-mean `h` defines the environmental regime; centre `water_depth_m` checks that the background column reaches 500 m. The upper/deep velocity difference is a layer contrast, not a vertical derivative. Do not compare it numerically to the derivative of the whole-column fitted tilt as a closed motion budget.

The analysis notebook uses trailing regime smoothing by default so future PV observations cannot set today's regime; centred smoothing is retained as a sensitivity. All settings and file fingerprints are stored beside the Parquet cache.